# ❤️ Heart Disease Detection — Data Analysis & Visualization
> **MA411 — Expert Systems Project**  
> Exploratory Data Analysis, Statistical Summary, and Visual Insights

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✔ Libraries loaded successfully')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/raw_data.csv')
print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

### Feature Dictionary

| Feature | Description | Type |
|---------|-------------|------|
| `age` | Age in years | Numeric |
| `sex` | 0 = Female, 1 = Male | Binary |
| `cp` | Chest pain type (0–3) | Categorical |
| `trestbps` | Resting blood pressure (mmHg) | Numeric |
| `chol` | Serum cholesterol (mg/dl) | Numeric |
| `fbs` | Fasting blood sugar > 120 mg/dl | Binary |
| `restecg` | Resting ECG results (0–2) | Categorical |
| `thalach` | Maximum heart rate achieved | Numeric |
| `exang` | Exercise induced angina (0/1) | Binary |
| `oldpeak` | ST depression by exercise | Numeric |
| `slope` | Slope of peak exercise ST segment | Categorical |
| `ca` | Major vessels colored (0–3) | Numeric |
| `thal` | Thalassemia (1–3) | Categorical |
| `target` | Heart disease (0=No, 1=Yes) | **Target** |

## 3. Statistical Summary

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', axis=1)

In [ ]:
# Missing values summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].style.highlight_max(color='#ffcccc')

## 4. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['target'].value_counts()
axes[0].bar(['No Disease', 'Disease'], counts.values,
            color=['#27ae60', '#c0392b'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Target Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['No Disease', 'Disease'],
            colors=['#27ae60', '#c0392b'], autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportions', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Class balance ratio: {counts[0]}/{counts[1]} = {counts[0]/counts[1]:.2f}')

## 5. Correlation Heatmap

In [ ]:
plt.figure(figsize=(13, 9))
corr = df.fillna(df.median()).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # Show lower triangle only
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Matrix\n(Lower Triangle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Feature Distributions by Target Class

In [ ]:
numeric_feats = ['age', 'chol', 'trestbps', 'thalach', 'oldpeak']
fig, axes = plt.subplots(1, len(numeric_feats), figsize=(18, 4))

for ax, feat in zip(axes, numeric_feats):
    for label, color, name in [(0, '#27ae60', 'No Disease'), (1, '#c0392b', 'Disease')]:
        data = df[df['target'] == label][feat].dropna()
        ax.hist(data, bins=20, alpha=0.6, color=color, label=name, edgecolor='white')
    ax.set_title(feat.upper(), fontweight='bold', fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Target Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Boxplots — Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
boxplot_feats = ['age', 'chol', 'trestbps', 'thalach', 'oldpeak', 'ca']

for ax, feat in zip(axes.flat, boxplot_feats):
    data_grouped = [df[df['target']==0][feat].dropna(),
                    df[df['target']==1][feat].dropna()]
    bp = ax.boxplot(data_grouped, labels=['No Disease', 'Disease'],
                    patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('#a8d5ba')
    bp['boxes'][1].set_facecolor('#f5a9a9')
    ax.set_title(f'{feat} Distribution', fontweight='bold')
    ax.set_ylabel(feat)

plt.suptitle('Boxplots — Feature Comparison by Target Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Categorical Feature Analysis

In [ ]:
cat_feats = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope']
labels_map = {
    'sex': {0: 'Female', 1: 'Male'},
    'fbs': {0: 'Normal', 1: 'High'},
    'exang': {0: 'No', 1: 'Yes'},
    'cp': {0: 'Typical', 1: 'Atypical', 2: 'Non-Anginal', 3: 'Asymptomatic'},
    'restecg': {0: 'Normal', 1: 'ST Abnorm.', 2: 'LVH'},
    'slope': {0: 'Upsloping', 1: 'Flat', 2: 'Downsloping'},
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, feat in zip(axes.flat, cat_feats):
    ct = pd.crosstab(df[feat], df['target'])
    ct.plot(kind='bar', ax=ax, color=['#27ae60', '#c0392b'],
            alpha=0.85, edgecolor='white', legend=False)
    
    if feat in labels_map:
        ax.set_xticklabels([labels_map[feat].get(int(t.get_text()), t.get_text())
                            for t in ax.get_xticklabels()], rotation=30, ha='right')
    ax.set_title(f'{feat.upper()} vs Target', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Count')

# Single legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#27ae60', label='No Disease'),
                   Patch(facecolor='#c0392b', label='Disease')]
fig.legend(handles=legend_elements, loc='upper right', fontsize=11)
plt.suptitle('Categorical Features vs Heart Disease', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Correlation with Target (Top Features)

In [ ]:
corr_target = df.fillna(df.median()).corr()['target'].drop('target').abs().sort_values(ascending=False)

colors = ['#c0392b' if c > 0 else '#27ae60'
          for c in df.fillna(df.median()).corr()['target'].drop('target')[corr_target.index]]

plt.figure(figsize=(10, 5))
bars = plt.barh(corr_target.index, corr_target.values, color=colors, alpha=0.85, edgecolor='white')
plt.xlabel('|Correlation| with Target', fontsize=12)
plt.title('Feature Importance by Correlation with Heart Disease', fontsize=13, fontweight='bold')
plt.gca().invert_yaxis()

for bar, val in zip(bars, corr_target.values):
    plt.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)

from matplotlib.patches import Patch
plt.legend(handles=[Patch(color='#c0392b', label='Positive corr'),
                    Patch(color='#27ae60', label='Negative corr')])
plt.tight_layout()
plt.show()

print('\nTop 5 most correlated features with heart disease:')
for feat, val in corr_target.head(5).items():
    print(f'  {feat:12s}: {val:.4f}')

## 10. Pairplot — Key Features

In [ ]:
key_cols = ['age', 'chol', 'thalach', 'oldpeak', 'target']
pair_df = df[key_cols].fillna(df[key_cols].median())
pair_df['target_label'] = pair_df['target'].map({0: 'No Disease', 1: 'Disease'})

g = sns.pairplot(pair_df.drop('target', axis=1),
                 hue='target_label',
                 palette={'No Disease': '#27ae60', 'Disease': '#c0392b'},
                 diag_kind='kde', plot_kws={'alpha': 0.5, 's': 30})
g.fig.suptitle('Pairplot of Key Numeric Features', y=1.02, fontsize=13, fontweight='bold')
plt.show()

## 11. Summary & Key Insights

| Insight | Finding |
|---------|----------|
| **Dataset** | 303 patients, 13 features, balanced classes (~47% / 53%) |
| **Missing Data** | 5 missing values in `trestbps`, `chol`, `thalach` — filled with median |
| **Age** | Disease patients tend to be older (mean ~56 vs ~52) |
| **Cholesterol** | High cholesterol correlates with disease risk |
| **Max HR** | Lower max heart rate strongly associated with disease |
| **ST Depression** | Higher oldpeak = higher risk |
| **Exercise Angina** | Strongest single binary predictor |
| **Chest Pain** | Asymptomatic type paradoxically most associated with disease |

> **Next Step** → See `model_training.ipynb` for Decision Tree training and Expert System comparison.